# TwinBeat — Arrhythmia Detection

Notebook-based ECG classification experiment using PTB-XL.

This is the final notebook supplied from Downloads, prepared for portfolio review.
The original experiment logic is retained, including its methodological limitations.
Paths and installation instructions are made portable; saved outputs are cleared.

**Read before interpreting results:** SMOTE and scaling occur before the split;
the split is not patient-grouped; `VA` is a catch-all label, not a verified ventricular
arrhythmia diagnosis. Historical scores are not reliable independent-test estimates.
This research prototype is not for clinical diagnosis.

See [README](README.md), [methodology review](docs/METHODOLOGY.md), and
[historical results](docs/RESULTS.md). No website or backend is included.


## Install dependencies

In [ ]:
%pip install -r requirements.txt

## Configure data and output paths

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast

from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("PTBXL_DATA_DIR", "data/ptb-xl"))
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for required in ["ptbxl_database.csv", "scp_statements.csv", "records500"]:
    if not (DATA_DIR / required).exists():
        raise FileNotFoundError(f"Missing {DATA_DIR / required}; see docs/DATASET.md")
path = str(DATA_DIR) + os.sep

# Load metadata
df = pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)

print("Total records:", len(df))
print(df.head(3))

## Historical label mapping

**Label caveat:** `SR` takes precedence even if other codes coexist. `AFIB` and `AFLT` are merged. Every other record receives `VA`; this residual group is not evidence of ventricular arrhythmia. The mapping below is preserved as historical code, not endorsed as a clinical taxonomy.

In [ ]:
# Load SCP statements to understand diagnosis codes
scp = pd.read_csv(path + 'scp_statements.csv', index_col=0)

# Function to assign labels: NSR, AFib, or VA
def assign_label(scp_codes):
    codes = list(scp_codes.keys())
    if 'SR' in codes:
        return 'NSR'
    elif 'AFIB' in codes or 'AFLT' in codes:
        return 'AFib'
    else:
        return 'VA'

df['label'] = df['scp_codes'].apply(assign_label)

print("Label distribution:")
print(df['label'].value_counts())

## Load 500 Hz ECG signals

In [ ]:
# Load ECG signals at 500Hz
def load_raw_data(df, path):
    data = []
    for f in df.filename_hr:
        signal, meta = wfdb.rdsamp(path + f)
        data.append(signal)
    return np.array(data)

print("Loading all ECG signals; memory usage and runtime depend on your machine.")
X_raw = load_raw_data(df, path)
y = df['label'].values

print("ECG data shape:", X_raw.shape)  # should be (21799, 5000, 12)
print("Labels shape:", y.shape)

## Initial NeuroKit feature-extraction prototype

In [ ]:
import neurokit2 as nk
from scipy.signal import welch
import warnings
warnings.filterwarnings('ignore')

def extract_features(signal_12lead, fs=500):
    features = {}
    
    for lead_idx in range(12):
        lead = signal_12lead[:, lead_idx]
        prefix = f'lead{lead_idx}'
        
        try:
            # --- TIME DOMAIN (RR interval based) ---
            peaks, _ = nk.ecg_peaks(lead, sampling_rate=fs)
            rpeaks = np.where(peaks['ECG_R_Peaks'] == 1)[0]
            
            if len(rpeaks) > 1:
                rr = np.diff(rpeaks) / fs * 1000  # in ms
                features[f'{prefix}_mean_rr'] = np.mean(rr)
                features[f'{prefix}_sdrr'] = np.std(rr)
                features[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(rr)**2))
                features[f'{prefix}_rr50'] = np.sum(np.abs(np.diff(rr)) > 50)
                features[f'{prefix}_prr50'] = features[f'{prefix}_rr50'] / len(rr) * 100
                features[f'{prefix}_hrv'] = np.max(rr) - np.min(rr)
            else:
                for feat in ['mean_rr','sdrr','rmssd','rr50','prr50','hrv']:
                    features[f'{prefix}_{feat}'] = 0

            # --- MORPHOLOGICAL FEATURE PROTOTYPE ---
            try:
                signals, info = nk.ecg_process(lead, sampling_rate=fs)
                
                # QRS duration
                qrs_on = info.get('ECG_Q_Peaks', [])
                qrs_off = info.get('ECG_S_Peaks', [])
                if len(qrs_on) > 0 and len(qrs_off) > 0:
                    n = min(len(qrs_on), len(qrs_off))
                    qrs_dur = (np.array(qrs_off[:n]) - np.array(qrs_on[:n])) / fs * 1000
                    features[f'{prefix}_qrs_mean'] = np.mean(qrs_dur)
                    features[f'{prefix}_qrs_std'] = np.std(qrs_dur)
                else:
                    features[f'{prefix}_qrs_mean'] = 0
                    features[f'{prefix}_qrs_std'] = 0

                # P-wave amplitude
                p_peaks = info.get('ECG_P_Peaks', [])
                p_peaks = [p for p in p_peaks if not np.isnan(p)]
                if len(p_peaks) > 0:
                    p_amps = lead[np.array(p_peaks, dtype=int)]
                    features[f'{prefix}_p_amp_mean'] = np.mean(p_amps)
                    features[f'{prefix}_p_amp_std'] = np.std(p_amps)
                else:
                    features[f'{prefix}_p_amp_mean'] = 0
                    features[f'{prefix}_p_amp_std'] = 0

                # T-wave amplitude
                t_peaks = info.get('ECG_T_Peaks', [])
                t_peaks = [t for t in t_peaks if not np.isnan(t)]
                if len(t_peaks) > 0:
                    t_amps = lead[np.array(t_peaks, dtype=int)]
                    features[f'{prefix}_t_amp_mean'] = np.mean(t_amps)
                    features[f'{prefix}_t_amp_std'] = np.std(t_amps)
                else:
                    features[f'{prefix}_t_amp_mean'] = 0
                    features[f'{prefix}_t_amp_std'] = 0

            except:
                for feat in ['qrs_mean','qrs_std','p_amp_mean','p_amp_std','t_amp_mean','t_amp_std']:
                    features[f'{prefix}_{feat}'] = 0

            # --- FREQUENCY DOMAIN ---
            freqs, psd = welch(lead, fs=fs, nperseg=1024)
            vlf = np.trapz(psd[(freqs >= 0.0033) & (freqs < 0.04)])
            lf  = np.trapz(psd[(freqs >= 0.04)   & (freqs < 0.15)])
            hf  = np.trapz(psd[(freqs >= 0.15)   & (freqs < 0.40)])
            features[f'{prefix}_vlf'] = vlf
            features[f'{prefix}_lf']  = lf
            features[f'{prefix}_hf']  = hf
            features[f'{prefix}_lf_hf_ratio'] = lf / hf if hf > 0 else 0

        except:
            for feat in ['mean_rr','sdrr','rmssd','rr50','prr50','hrv',
                         'qrs_mean','qrs_std','p_amp_mean','p_amp_std',
                         't_amp_mean','t_amp_std','vlf','lf','hf','lf_hf_ratio']:
                features[f'{prefix}_{feat}'] = 0

    return features

# Test on one record first
print("Testing feature extraction on 1 record...")
test_features = extract_features(X_raw[0])
print(f"Number of features extracted: {len(test_features)}")
print("Sample features:", list(test_features.keys())[:10])

## Fast feature extraction used for model training

In [ ]:
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def extract_features_fast(signal_12lead, fs=500):
    features = {}
    
    for lead_idx in range(12):
        lead = signal_12lead[:, lead_idx]
        prefix = f'lead{lead_idx}'
        
        try:
            # TIME DOMAIN - fast version using scipy only
            from scipy.signal import find_peaks
            # Find R peaks simply using find_peaks (much faster than neurokit)
            peaks, _ = find_peaks(lead, distance=fs*0.3, height=np.mean(lead))
            
            if len(peaks) > 1:
                rr = np.diff(peaks) / fs * 1000
                features[f'{prefix}_mean_rr'] = np.mean(rr)
                features[f'{prefix}_sdrr'] = np.std(rr)
                features[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(rr)**2))
                features[f'{prefix}_rr50'] = np.sum(np.abs(np.diff(rr)) > 50)
                features[f'{prefix}_prr50'] = features[f'{prefix}_rr50'] / len(rr) * 100
                features[f'{prefix}_hrv'] = np.max(rr) - np.min(rr)
                
                # MORPHOLOGICAL - amplitude based (fast)
                r_amps = lead[peaks]
                features[f'{prefix}_r_amp_mean'] = np.mean(r_amps)
                features[f'{prefix}_r_amp_std'] = np.std(r_amps)
                
                # ST segment approximation (after R peak)
                st_amps = []
                for p in peaks:
                    st_idx = p + int(0.08 * fs)  # ~80ms after R peak
                    if st_idx < len(lead):
                        st_amps.append(lead[st_idx])
                features[f'{prefix}_st_mean'] = np.mean(st_amps) if st_amps else 0
                features[f'{prefix}_st_std'] = np.std(st_amps) if st_amps else 0
                
                # T wave approximation (after R peak)
                t_amps = []
                for p in peaks:
                    t_idx = p + int(0.2 * fs)  # ~200ms after R peak
                    if t_idx < len(lead):
                        t_amps.append(lead[t_idx])
                features[f'{prefix}_t_amp_mean'] = np.mean(t_amps) if t_amps else 0
                features[f'{prefix}_t_amp_std'] = np.std(t_amps) if t_amps else 0
                
                # P wave approximation (before R peak)
                p_amps = []
                for p in peaks:
                    p_idx = p - int(0.15 * fs)  # ~150ms before R peak
                    if p_idx >= 0:
                        p_amps.append(lead[p_idx])
                features[f'{prefix}_p_amp_mean'] = np.mean(p_amps) if p_amps else 0
                features[f'{prefix}_p_amp_std'] = np.std(p_amps) if p_amps else 0

            else:
                for feat in ['mean_rr','sdrr','rmssd','rr50','prr50','hrv',
                            'r_amp_mean','r_amp_std','st_mean','st_std',
                            't_amp_mean','t_amp_std','p_amp_mean','p_amp_std']:
                    features[f'{prefix}_{feat}'] = 0

            # FREQUENCY DOMAIN
            freqs, psd = welch(lead, fs=fs, nperseg=512)  # smaller nperseg = faster
            vlf = np.trapz(psd[(freqs >= 0.0033) & (freqs < 0.04)])
            lf  = np.trapz(psd[(freqs >= 0.04)   & (freqs < 0.15)])
            hf  = np.trapz(psd[(freqs >= 0.15)   & (freqs < 0.40)])
            features[f'{prefix}_vlf'] = vlf
            features[f'{prefix}_lf']  = lf
            features[f'{prefix}_hf']  = hf
            features[f'{prefix}_lf_hf_ratio'] = lf / hf if hf > 0 else 0

        except:
            for feat in ['mean_rr','sdrr','rmssd','rr50','prr50','hrv',
                        'r_amp_mean','r_amp_std','st_mean','st_std',
                        't_amp_mean','t_amp_std','p_amp_mean','p_amp_std',
                        'vlf','lf','hf','lf_hf_ratio']:
                features[f'{prefix}_{feat}'] = 0
    
    return features

# Test on one record first
test = extract_features_fast(X_raw[0])
print(f"Features per record: {len(test)}")
print("Looks good! Now running on all records...")

# Run on full dataset
all_features = []
for i in tqdm(range(len(X_raw))):
    feats = extract_features_fast(X_raw[i])
    all_features.append(feats)

feature_df = pd.DataFrame(all_features)

# Save immediately
feature_df.to_csv(str(OUTPUT_DIR / 'extracted_features.csv'), index=False)
np.save(str(OUTPUT_DIR / 'labels.npy'), y)

print("Done! Shape:", feature_df.shape)

## Historical resampling and scaling

**Historical limitation:** resampling and fitting the scaler before the split leaks information and allows synthetic samples into evaluation. A corrected study must split patients first, fit preprocessing/resampling only within training folds, and rerun all models. The existing scores cannot be reused for that corrected study.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Load features (if needed)
X = feature_df.values
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)
print("Encoded:", np.unique(y_encoded))

# Handle missing/infinite values
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Check class distribution before SMOTE
print("\nBefore SMOTE:")
unique, counts = np.unique(y_encoded, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f"  {cls}: {cnt}")

# Apply SMOTE
print("\nApplying SMOTE...")
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y_encoded)

print("\nAfter SMOTE:")
unique, counts = np.unique(y_resampled, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f"  {cls}: {cnt}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)

print("\nFinal dataset shape:", X_scaled.shape)

## Historical random train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_resampled, 
    test_size=0.2, 
    random_state=42,
    stratify=y_resampled
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTrain class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f"  {cls}: {cnt}")

## Compare six classifiers

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import time

results = {}

models = {
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42),
    'LightGBM': lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
}

for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.time() - start
    
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, y_pred) * 100, 2),
        'F1': round(f1_score(y_test, y_pred, average='weighted') * 100, 2),
        'Precision': round(precision_score(y_test, y_pred, average='weighted') * 100, 2),
        'Recall': round(recall_score(y_test, y_pred, average='weighted') * 100, 2),
    }
    print(f"  ✓ Accuracy: {results[name]['Accuracy']}% | Time: {elapsed:.1f}s")

print("\nAll models done!")


## Train the stacking ensemble

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [
    ('xgb', XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lgbm', lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1))
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Ensemble...")
start = time.time()
stack.fit(X_train, y_train)
y_pred_stack = stack.predict(X_test)
elapsed = time.time() - start

stack_acc = round(accuracy_score(y_test, y_pred_stack) * 100, 2)
stack_f1 = round(f1_score(y_test, y_pred_stack, average='weighted') * 100, 2)
stack_prec = round(precision_score(y_test, y_pred_stack, average='weighted') * 100, 2)
stack_rec = round(recall_score(y_test, y_pred_stack, average='weighted') * 100, 2)

print(f"✓ Stacking Accuracy: {stack_acc}%")
print(f"✓ F1: {stack_f1}%")
print(f"✓ Precision: {stack_prec}%")
print(f"✓ Recall: {stack_rec}%")
print(f"✓ Time: {elapsed:.1f}s")

## Summarize metrics and plot the confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get full metrics for all models
print("="*60)
print("COMPLETE RESULTS SUMMARY")
print("="*60)
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  Accuracy:  {metrics['Accuracy']}%")
    print(f"  F1-Score:  {metrics['F1']}%")
    print(f"  Precision: {metrics['Precision']}%")
    print(f"  Recall:    {metrics['Recall']}%")

print(f"\nStacking Ensemble:")
print(f"  Accuracy:  {stack_acc}%")
print(f"  F1-Score:  {stack_f1}%")
print(f"  Precision: {stack_prec}%")
print(f"  Recall:    {stack_rec}%")

# Confusion matrix for best model (Stacking)
cm = confusion_matrix(y_test, y_pred_stack)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title(f'Confusion Matrix - Stacking Ensemble ({stack_acc}%)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()
print("Confusion matrix saved!")

## Micro-averaged multiclass ROC curves

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Binarize for ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

plt.figure(figsize=(10, 7))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'black']
model_list = [
    ('XGBoost', models['XGBoost']),
    ('Random Forest', models['Random Forest']),
    ('SVM', models['SVM']),
    ('KNN', models['KNN']),
    ('AdaBoost', models['AdaBoost']),
    ('LightGBM', models['LightGBM']),
    ('Stacking', stack)
]

for (name, model), color in zip(model_list, colors):
    y_score = model.predict_proba(X_test)
    fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'roc_curves.png'), dpi=150)
plt.show()
print("ROC curve saved!")

## Exploratory feature importance

**Interpretation:** the model importance scales differ (for example, LightGBM defaults to split counts). Their raw average is an exploratory graphic, not a calibrated cross-model or causal explanation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get feature importance from the RF base estimator inside the stack
rf_model = stack.named_estimators_['rf']
xgb_model = stack.named_estimators_['xgb']
lgbm_model = stack.named_estimators_['lgbm']

# Average importance across all three base models
rf_imp = rf_model.feature_importances_
xgb_imp = xgb_model.feature_importances_
lgbm_imp = lgbm_model.feature_importances_

avg_imp = (rf_imp + xgb_imp + lgbm_imp) / 3
feature_names = feature_df.columns.tolist()

# Top 20
top_idx = np.argsort(avg_imp)[-20:][::-1]
top_features = [feature_names[i] for i in top_idx]
top_values = avg_imp[top_idx]

plt.figure(figsize=(10, 8))
plt.barh(top_features[::-1], top_values[::-1], color='steelblue')
plt.xlabel('Average Feature Importance')
plt.title('Top 20 Most Important Features (Averaged Across Ensemble)')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'feature_importance.png'), dpi=150)
plt.show()
print("Saved!")

## Export fitted models and preprocessing objects

In [ ]:
import joblib

# Save all 7 models
joblib.dump(models['XGBoost'], str(OUTPUT_DIR / 'xgboost_model.pkl'))
joblib.dump(models['Random Forest'], str(OUTPUT_DIR / 'rf_model.pkl'))
joblib.dump(models['SVM'], str(OUTPUT_DIR / 'svm_model.pkl'))
joblib.dump(models['KNN'], str(OUTPUT_DIR / 'knn_model.pkl'))
joblib.dump(models['AdaBoost'], str(OUTPUT_DIR / 'adaboost_model.pkl'))
joblib.dump(models['LightGBM'], str(OUTPUT_DIR / 'lgbm_model.pkl'))
joblib.dump(stack, str(OUTPUT_DIR / 'stacking_model.pkl'))

# Save the scaler and label encoder too
joblib.dump(scaler, str(OUTPUT_DIR / 'scaler.pkl'))
joblib.dump(le, str(OUTPUT_DIR / 'label_encoder.pkl'))

print("All models saved!")

## Inspect exported file sizes

In [ ]:
import os

files = ['xgboost_model.pkl', 'rf_model.pkl', 'svm_model.pkl', 
         'knn_model.pkl', 'adaboost_model.pkl', 'lgbm_model.pkl', 
         'stacking_model.pkl', 'scaler.pkl', 'label_encoder.pkl']

for f in files:
    export_path = OUTPUT_DIR / f
    size = os.path.getsize(export_path) / (1024*1024)
    print(f'{f}: {size:.1f} MB')

## Save feature-column order

In [ ]:
import joblib
import json

# Save feature column names (needed to align features correctly)
feature_names = list(feature_df.columns)
joblib.dump(feature_names, str(OUTPUT_DIR / 'feature_names.pkl'))

# Verify everything is saved
print("Feature names count:", len(feature_names))
print("First 5 features:", feature_names[:5])
print("Label classes:", le.classes_)
print("\nArtifacts saved. No backend is included in this repository.")